# core

> Fill in a module description here

In [ ]:
# | default_exp api

In [ ]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [ ]:
import os
import subprocess
import sys
from dotenv import load_dotenv

if os.getenv("USER") == "max":
    # Get the current working directory
    current_dir = os.getcwd()

    # Append the parent directory to sys.path
    parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
    sys.path.append(parent_dir)

    # Decrypt the .env.local file and load variables
    try:
        # Decrypt and output to stdout
        os.system(f"dotenvx decrypt -f {os.path.abspath(os.path.join(current_dir, '..'))}/.env.local")

        # Load the decrypted environment variables from stdout
        load_dotenv(dotenv_path=os.path.abspath(os.path.join(current_dir, '..')) + "/.env.local")

        os.system(f"dotenvx encrypt -f {os.path.abspath(os.path.join(current_dir, '..'))}/.env.local")

        print("Environment variables loaded successfully.")
        os.environ["DOCKER_HOST"] = "unix:///Users/max/.docker/run/docker.sock"
    except subprocess.CalledProcessError as e:
        print(f"Error decrypting .env.local: {e}")
        sys.exit(1)

In [1]:
# | export
from typing import Any, List, TypedDict, Tuple, Sequence
from product_horse.filesystems import (
    AbstractFileSystem,
    FilePathType,
)
from product_horse.db import (
    AbstractDatabase,
    DBType,
    UnvalidatedUser,
    UnvalidatedFileMetadata,
    User,
    FileMetadata,
    UnvalidatedTranscription,
    Transcription,
    Schema,
    UnvalidatedSchema,
    UnvalidatedUtterance,
    Utterance,
    UtteranceSegment,
    Employee,
    Video,
)
from product_horse.audio import (
    AudioEditor,
)
from product_horse.search import SearchEngine

ModuleNotFoundError: No module named 'product_horse'

In [ ]:
# | export


class FileDict(TypedDict):
    content: bytes
    name: str


def _create_user_and_add_files(
    user_data: dict[str, Any],
    files: List[FileDict],
    db: AbstractDatabase[DBType],
    fs: AbstractFileSystem,
    employee: Employee,
) -> Tuple[User, List[FileMetadata]]:
    """
    Create a new user and add files for the user.

    Args:
        user_data (dict[str, Any]): User data dictionary.
        files (List[FileDict]): List of file dictionaries containing content and name.
        db (AbstractDatabase): Database instance for saving user and file metadata.
        fs (AbstractFileSystem): File system instance for saving files.
        authorized (bool, optional): Whether the files are authorized. Defaults to True.

    Returns:
        Tuple[User, List[FileMetadata]]: The created user object and a list of file metadata objects created.
    """
    unvalidated_user = UnvalidatedUser(**user_data)
    user: User = db.save_user(unvalidated_user)
    metadata: List[FileMetadata] = []

    for file_info in files:
        file_path = fs.build_user_path(user, FilePathType.USER_ID_BASE)
        fs.create_folder(file_path)
        file_content = file_info["content"]
        file = fs.create_file(
            file_path, file_content, file_info["name"], authorized=True
        )
        # Save file metadata to database
        unvalidated_metadata = UnvalidatedFileMetadata(
            user_id=user.id,
            file_name=file_info["name"],
            file_path=file.uri,
            created_by_id=employee.id,
            company_id=employee.company_id,
        )
        metadata.append(
            db.as_employee(employee).save_file_metadata(unvalidated_metadata)
        )

    return user, metadata

In [ ]:
# | export


async def _transcribe_file_and_extract_speakers(
    file_metadata: FileMetadata,
    db: AbstractDatabase[DBType],
    audio_editor: AudioEditor,
    employee: Employee,
) -> Tuple[Transcription, List[str]]:
    """
    Transcribe an audio file, extract speaker names, and save the transcription to the database.

    Args:
        file_metadata (FileMetadata): The metadata of the file to transcribe.
        db (AbstractDatabase): Database instance for saving transcription data.
        audio_editor (AssemblyAIClient): AssemblyAI client instance for transcription.

    Returns:
        Tuple[Transcription, List[str]]: The created transcription object and a list of extracted speaker names.
    """
    transcription_result = await audio_editor.transcribe(
        str(file_metadata.file_path), employee
    )
    utterances: List[UnvalidatedUtterance] = transcription_result.utterances
    speaker_names: List[str] = list(
        set([utterance.speaker for utterance in utterances])
    )

    # Future - update speaker names in the transcript using AI + find and replace

    # Save transcription to the database
    unvalidated_transcription = UnvalidatedTranscription(
        file_id=file_metadata.id,
        text=transcription_result.text,
        company_id=employee.company_id,
        created_by_id=employee.id,
    )
    transcription: Transcription = db.as_employee(employee).save_transcription(
        unvalidated_transcription, utterances
    )

    return transcription, speaker_names

In [ ]:
# # | export
# async def create_and_save_schema(
#     text: str,
#     db: AbstractDatabase[DBType],
#     ai_model_client: AIModelClient,
#     employee: Employee,
# ) -> Schema:
#     """
#     Create a schema from the given text using an AI model and save it to the database.

#     Args:
#         text (str): The text to create a schema from.
#         db (AbstractDatabase): The database instance.
#         ai_model_client (AIModelClient): The AI model client instance.

#     Returns:
#         Schema: The saved schema object.
#     """
#     questions = await ai_model_client.create_schema(text)
#     unvalidated_schema = UnvalidatedSchema(
#         input_text=text,
#         json_schema=questions.model_dump(),
#         company_id=employee.company_id,
#         created_by_id=employee.id,
#     )
#     schema = db.as_employee(employee).save_schema(unvalidated_schema)
#     return schema

In [ ]:
# # | export

# import asyncio


# async def extract_info_from_transcriptions(
#     schema: Schema,
#     transcriptions: Sequence[Transcription],
#     db: AbstractDatabase[DBType],
#     ai_model_client: AIModelClient,
# ) -> Sequence[Answers]:
#     """
#     Extract information from transcriptions based on a schema and yields each extraction.

#     Args:
#         schema (Schema): The schema to use for extraction. Converted to questions internally.
#         transcriptions (Sequence[Transcription]): The list of transcriptions to extract information from.
#         db (AbstractDatabase): The database instance.
#         ai_model_client (AIModelClient): The AI model client instance.

#     Yields:
#         Iterator[Tuple[Transcription, dict]]: An iterator of tuples containing the transcription and extracted answers.
#     """
#     questions = Questions(**schema.json_schema)

#     tasks = [
#         ai_model_client.extract_information(transcription.text, questions)
#         for transcription in transcriptions
#     ]

#     # Use asyncio.gather to run them concurrently and wait for all to complete
#     answers: list[Answers] = await asyncio.gather(*tasks)

#     return answers

In [ ]:
# | export
async def get_relevant_utterances_from_query(
    query: str,
    transcripts: Sequence[Transcription],
    db: AbstractDatabase[DBType],
    employee: Employee,
) -> Sequence[Utterance]:
    """
    Returns time-sorted utterances from a query. Really shitty, needs fixing.
    """
    # allow for several clips from each transcript
    clips_requested = int(max(len(transcripts) * 2, 50))
    search_engine = SearchEngine(
        seconds_buffer=2, similarity_top_k=clips_requested, db=db, employee=employee
    )
    utterances = await search_engine.get_utterances_from_query(query, transcripts)

    return utterances

In [ ]:
# | export
async def save_files_and_transcriptions(
    user_id: str,
    user_name: str,
    file_paths: list[str],
    db: AbstractDatabase[DBType],
    fs: AbstractFileSystem,
    audio_editor: AudioEditor,
    employee: Employee,
):
    """Save the files and transcriptions to the database."""
    if user_id is None:  # type: ignore
        raise ValueError("User ID is required")
    user_data = {"external_id": user_id, "name": user_name}
    file_dicts: list[FileDict] = []
    for file in file_paths:
        with open(file, "rb") as f:  # Open the file in binary mode
            content = f.read()
        file_dicts.append({"content": content, "name": file})
    _, metadata = _create_user_and_add_files(user_data, file_dicts, db, fs, employee)
    transcriptions: list[Tuple[Transcription, list[str]]] = []
    for file_metadata in metadata:
        transcription = await _transcribe_file_and_extract_speakers(
            file_metadata, db, audio_editor, employee
        )
        transcriptions.append(transcription)
    return f"{len(transcriptions)} transcriptions saved."

In [ ]:
# # | export
# async def extract_data_given_users(
#     user_ids: List[str],
#     db: AbstractDatabase[DBType],
#     ai_model_client: AIModelClient,
#     app_working_directory: str,
# ):
#     schema = db.get_latest_schema()
#     if schema is None:
#         raise ValueError("No schema found")
#     questions = Questions(**schema.json_schema)
#     transcriptions = db.get_transcriptions_by_user_ids(user_ids)
#     results = await extract_info_from_transcriptions(
#         schema, transcriptions, db, ai_model_client
#     )

#     qa_data_manager = QADataManager()

#     for transcription, answers in zip(transcriptions, results):
#         metadata = db.get_file_metadata(transcription.file_id)
#         if metadata is None:
#             raise ValueError(f"No metadata found for file_id {transcription.file_id}")
#         user = db.get_user(metadata.user_id)
#         if user is None:
#             raise ValueError(f"No user found for user_id {metadata.user_id}")
#         qa_data_manager.add_answers(
#             questions,
#             answers,
#             document_name=str(metadata.file_name),
#             user_id=str(user.name),
#         )

#     output_file_path = f"{app_working_directory}/output.csv"
#     qa_data_manager.merge_rows_by_user()
#     qa_data_manager.to_csv(file_path=output_file_path)
#     return output_file_path

In [ ]:
# | export


def get_user_names_and_transcript_counts(
    db: AbstractDatabase[DBType],
) -> Sequence[tuple[str, str]]:
    """Only relevant for gradio"""
    users = db.get_all_users()
    user_ids = [user.id for user in users]
    if len(user_ids) == 0:
        return []
    last_user_id = user_ids[-1]
    options: list[tuple[str, str]] = []
    for user in users:
        transcriptions = db.get_transcriptions(user.id)
        options.append(
            (f"{user.name} ({len(transcriptions)}) calls transcribed", str(user.id))
        )
    assert str(last_user_id) == options[-1][1]
    return options

In [ ]:
# | hide
import nbdev # type: ignore

nbdev.nbdev_export()  # type: ignore